# Baseline Model — Street Hail Demand Forecasting

**Goal**: Establish a naive baseline to beat before touching LightGBM.

We test three baselines:
1. **Historical Mean** — predict the average trip count for this zone × hour × day-of-week
2. **Last Week Same Slot** — predict using the same zone × timeslot exactly 1 week ago
3. **Last 4 Weeks Average** — rolling average of the same slot over the past 4 weeks

Metrics: **MAE**, **RMSE**, **MAPE** (on zones with meaningful volume)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

ROOT        = Path('..')
DEMAND_PATH = ROOT / 'data' / 'processed' / 'demand_enriched.parquet'  # cleaned + zero-filled
LOOKUP      = ROOT / 'Meta Data' / 'Lookups' / 'taxi_zone_lookup.csv'

## 1. Load Demand Table

In [ ]:
# demand_enriched already has only active zones (filtered in 02b) and all temporal
# columns pre-built — just load and use directly, no re-filtering or re-extraction needed
demand = pd.read_parquet(DEMAND_PATH)
demand['time_bucket'] = pd.to_datetime(demand['time_bucket'])

print(f'Rows: {len(demand):,}')
print(f'Date range: {demand["time_bucket"].min().date()} → {demand["time_bucket"].max().date()}')
print(f'Zones: {demand["PULocationID"].nunique()}')
demand.head(3)

## 2. Train / Test Split

Use the last 4 weeks as the test set — everything before is training.

In [ ]:
cutoff = demand['time_bucket'].max() - pd.Timedelta(weeks=4)

train = demand[demand['time_bucket'] <= cutoff].copy()
test  = demand[demand['time_bucket'] >  cutoff].copy()

print(f'Train: {train["time_bucket"].min().date()} → {train["time_bucket"].max().date()}  ({len(train):,} rows)')
print(f'Test:  {test["time_bucket"].min().date()}  → {test["time_bucket"].max().date()}   ({len(test):,} rows)')

cutoff = demand['time_bucket'].max() - pd.Timedelta(weeks=4)

# No .copy() — train/test are read-only views until we add prediction columns
train = demand[demand['time_bucket'] <= cutoff]
test  = demand[demand['time_bucket'] >  cutoff].copy()   # copy only test — we'll add columns to it

print(f'Train: {train["time_bucket"].min().date()} → {train["time_bucket"].max().date()}  ({len(train):,} rows)')
print(f'Test:  {test["time_bucket"].min().date()} → {test["time_bucket"].max().date()}   ({len(test):,} rows)')

In [ ]:
train['hour']      = train['time_bucket'].dt.hour
train['minute']    = train['time_bucket'].dt.minute
train['dayofweek'] = train['time_bucket'].dt.dayofweek

hist_mean = (
    train.groupby(['PULocationID', 'dayofweek', 'hour', 'minute'])['trip_count']
    .mean()
    .reset_index()
    .rename(columns={'trip_count': 'pred_hist_mean'})
)

test = test.copy()
test['hour']      = test['time_bucket'].dt.hour
test['minute']    = test['time_bucket'].dt.minute
test['dayofweek'] = test['time_bucket'].dt.dayofweek

test = test.merge(hist_mean, on=['PULocationID', 'dayofweek', 'hour', 'minute'], how='left')
test['pred_hist_mean'] = test['pred_hist_mean'].fillna(test['trip_count'].mean())
print('Baseline 1 ready.')

# hour, minute, dayofweek already exist in demand_enriched — no re-extraction needed.
# Use slot_of_day (= hour*4 + minute//15) as the groupby key — finer than hour alone.
hist_mean = (
    train.groupby(['PULocationID', 'dayofweek', 'slot_of_day'])['trip_count']
    .mean()
    .rename('pred_hist_mean')
)

# Map onto test via MultiIndex — no merge, no copy of full demand
test_idx = pd.MultiIndex.from_arrays(
    [test['PULocationID'], test['dayofweek'], test['slot_of_day']]
)
test['pred_hist_mean'] = hist_mean.reindex(test_idx).values
test['pred_hist_mean'] = test['pred_hist_mean'].fillna(train['trip_count'].mean())
print('Baseline 1 ready.')

In [ ]:
# Build lookup: for each (zone, time_bucket) in test, look up the same slot 7 days prior
test['lookup_key'] = test['time_bucket'] - pd.Timedelta(weeks=1)

last_week_lookup = demand[['PULocationID', 'time_bucket', 'trip_count']].copy()
last_week_lookup.columns = ['PULocationID', 'lookup_key', 'pred_last_week']

test = test.merge(last_week_lookup, on=['PULocationID', 'lookup_key'], how='left')
test['pred_last_week'] = test['pred_last_week'].fillna(test['pred_hist_mean'])
print('Baseline 2 ready.')

# Build a (zone, time_bucket) → trip_count index once — reused for B2 and B3.
# O(1) lookup per row vs O(n log n) merge.
demand_lookup = demand.set_index(['PULocationID', 'time_bucket'])['trip_count']

test_pu = test['PULocationID'].values
test_tb = test['time_bucket'].to_numpy()

def lookup_weeks_prior(n_weeks: int) -> np.ndarray:
    offset = np.timedelta64(n_weeks * 7, 'D')
    keys   = pd.MultiIndex.from_arrays([test_pu, test_tb - offset])
    return demand_lookup.reindex(keys).to_numpy(dtype=np.float32)

test['pred_last_week'] = lookup_weeks_prior(1)
test['pred_last_week'] = pd.Series(test['pred_last_week']).fillna(test['pred_hist_mean']).values
print('Baseline 2 ready.')

In [ ]:
preds_4w = []
for w in [1, 2, 3, 4]:
    lookup = demand[['PULocationID', 'time_bucket', 'trip_count']].copy()
    lookup['lookup_key'] = lookup['time_bucket'] + pd.Timedelta(weeks=w)
    lookup = lookup.rename(columns={'trip_count': f'w{w}'})
    preds_4w.append(lookup[['PULocationID', 'lookup_key', f'w{w}']])

merged = test[['PULocationID', 'time_bucket']].copy()
merged = merged.rename(columns={'time_bucket': 'lookup_key'})
for df_w in preds_4w:
    merged = merged.merge(df_w, on=['PULocationID', 'lookup_key'], how='left')

test['pred_4w_avg'] = merged[['w1','w2','w3','w4']].mean(axis=1)
test['pred_4w_avg'] = test['pred_4w_avg'].fillna(test['pred_hist_mean'])
print('Baseline 3 ready.')

# Reuse demand_lookup from Baseline 2 — no copies of demand, no merges.
# Stack 4 week-prior lookups into a (4 × n_test) array, nanmean across weeks.
week_preds = np.stack([lookup_weeks_prior(w) for w in range(1, 5)], axis=0)  # shape (4, n_test)
test['pred_4w_avg'] = np.nanmean(week_preds, axis=0)
test['pred_4w_avg'] = pd.Series(test['pred_4w_avg']).fillna(test['pred_hist_mean']).values
print('Baseline 3 ready.')

In [ ]:
def evaluate(y_true, y_pred, name):
    mae  = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    # MAPE — only where y_true > 0
    mask = y_true > 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    return {'Model': name, 'MAE': round(mae, 3), 'RMSE': round(rmse, 3), 'MAPE (%)': round(mape, 2)}

y_true = test['trip_count'].values

results = pd.DataFrame([
    evaluate(y_true, test['pred_hist_mean'].values, 'Historical Mean'),
    evaluate(y_true, test['pred_last_week'].values, 'Last Week Same Slot'),
    evaluate(y_true, test['pred_4w_avg'].values,    '4-Week Rolling Avg'),
])

results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = ['MAE', 'RMSE', 'MAPE (%)']
colors  = ['steelblue', 'coral', 'mediumseagreen']

for ax, metric, color in zip(axes, metrics, colors):
    ax.bar(results['Model'], results[metric], color=color)
    ax.set_title(metric)
    ax.set_xticklabels(results['Model'], rotation=15, ha='right', fontsize=9)

plt.suptitle('Baseline Model Comparison', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 7. Error Distribution — Best Baseline

In [ ]:
# Identify best baseline by MAE
best_col = results.loc[results['MAE'].idxmin(), 'Model']
col_map  = {
    'Historical Mean':    'pred_hist_mean',
    'Last Week Same Slot':'pred_last_week',
    '4-Week Rolling Avg': 'pred_4w_avg'
}
best_pred_col = col_map[best_col]
print(f'Best baseline: {best_col}')

errors = test['trip_count'] - test[best_pred_col]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(errors.clip(-50, 50), bins=60, color='steelblue', edgecolor='none')
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title(f'Residuals — {best_col} (clipped ±50)')
axes[0].set_xlabel('Actual − Predicted')

# Actual vs predicted scatter (sample 5000 points)
sample = test.sample(min(5000, len(test)), random_state=42)
axes[1].scatter(sample['trip_count'], sample[best_pred_col], alpha=0.3, s=5, color='coral')
lim = max(sample['trip_count'].max(), sample[best_pred_col].max())
axes[1].plot([0, lim], [0, lim], 'k--', linewidth=1)
axes[1].set_xlabel('Actual')
axes[1].set_ylabel('Predicted')
axes[1].set_title('Actual vs Predicted')

plt.tight_layout()
plt.show()

In [ ]:
# Save baseline results for comparison in LightGBM notebook
results.to_csv(ROOT / 'data' / 'processed' / 'baseline_results.csv', index=False)
print('Saved baseline_results.csv')
print('\n>>> Target to beat with LightGBM:')
print(results.to_string(index=False))